# 9주차 1차시: Adaptive RAG (쿼리 분석 기반 적응형 검색)

| 주제 | 내용 |
|---|---|
| LangGraph 복습 | SimpleState, TextState, 다중 노드 직렬 |
| Adaptive RAG 개념 | 질문 복잡도별 검색 전략 동적 결정 |
| 쿼리 분석 | QueryAnalysis, with_structured_output, 라우팅 분류 |
| Fallback 전략 | RoutingConfig, RoutingEngine, 신뢰도 기반 폴백 |
| 핸들러 패턴 | RouteHandler ABC, Direct/RAG/WebSearch 핸들러 |
| LangGraph 통합 | AdaptiveRAGState, 조건부 분기 워크플로 |

In [6]:
!pip install langgraph

In [5]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.0 which is incompatible.


## LangGraph 복습

In [8]:
import os
from typing import TypedDict, Annotated
# from dotenv import load_dotenv
# load_dotenv()

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage, SystemMessage, AIMessage, ToolMessage, BaseMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
import pandas as pd
llm = ChatOpenAI(model="gpt-4o-mini")

In [9]:
class SimpleState(TypedDict):
  message : str

def greet(state : SimpleState) -> dict:
  return {'message' : f'안녕, {state['message']}'}

builder = StateGraph(SimpleState)
builder.add_node('greet', greet)
builder.add_edge(START, 'greet')
builder.add_edge('greet', END)

app = builder.compile()

In [10]:
app.invoke({'message' : 'abc'})

{'message': '안녕, abc'}

## Adaptive RAG 개념

- 참고: https://arxiv.org/pdf/2403.14403
- 질문 복잡도에 따라 서로 다른 검색 전략을 선택하는 적응형 질의응답 프레임워크
- 모든 질문에 동일한 검색 절차를 적용하는 기존 RAG의 한계 해결
- 질문을 난이도별로 분류 → 검색 없이 답변 / 단일 검색 / 다단계 검색을 동적 결정
- 쉬운 질문에는 빠르고 가벼운 응답, 어려운 질문에는 깊은 정보 탐색

In [12]:
app.invoke({'text' : 'hello langgraph', 'upper': '', 'length' : 0})

{'text': 'hello langgraph', 'upper': 'HELLO LANGGRAPH', 'length': 15}

### Adaptive RAG 참고
- https://arxiv.org/pdf/2403.14403
- Adaptive-RAG는 질문 복잡도에 따라 서로 다른 검색 전략을 선택하는 적응형 질의응답 프레임워크,
- 기존 RAG는 외부 문서를 활용해 LLM의 한계를 보완하지만, 모든 질문에 동일한 검색 절차를 적용하면 단순 질의에는 비효율적이고 복잡한 질의에는 충분하지 않을 수 있음.
- Adaptive-RAG는 이러한 한계를 해결하기 위해 질문을 난이도별로 분류하고, 검색 없이 답변할지, 단일 검색을 사용할지, 다단계 검색을 수행할지를 동적으로 결정.
  - 이를 통해 쉬운 질문에는 빠르고 가벼운 응답을 제공하고, 어려운 질문에는 더 깊은 정보 탐색을 수행함으로써 정확도와 효율성을 동시에 높임.

### 쿼리 분석 (QueryAnalysis & with_structured_output)

- `BaseModel`로 출력 스키마 정의 → `llm.with_structured_output()`로 형식 강제
- `Literal` 타입으로 라우팅 경로를 제한 (direct / rag / web_search)

In [13]:
# 예전
  # T5 -> finetuning
  # query -> T5 -> 검색 필요없는것이면 simple 검색, multi-hop 검색
  # multi-hop 검색은 그래프 래그가 잘함
  # 예:내가 갔던 식당 셰프와 같은 식당에서 일했던 사람이 운영하는 식당들을 찾아주고 메뉴를 추천
    # 내가 갔던 식당은 --다.
    # 식당에서 일했던 사람(셰프)은 --다.
    # -- 이라는 사람은 -- 셰프와 같이 일했다.

# 지금
  # gpt, qen 등, 모델들이 파인튜닝이 필요없을 정도로 성능이 우수
  # Adaptive Rag 예
   # query -> chatbot -> 검색 필요없는 것 : direct 답변
                        # 상세검색 필요한것: RAG 검색
                        # 툴 필요? tool 호출

In [14]:
from typing import TypedDict, Annotated, Literal, List, Optional

In [16]:
class QueryAnalysis(BaseModel):
  query : str = Field(description = '원본 쿼리')
  route : Literal["direct", "rag", "web_search"] = Field( # 리터럴이란 route는 3개 중 하나의 값만 가질 수 있다는 것
            description = "라우팅 경로: direct(직접 답변), rag(문서 검색), web_search(웹 검색)"
  )
  confidence : float = Field(description = '분류 신뢰도(0.0 ~ 1.0)', ge=0.0, le=1.0) # 0.0보다 크거나 같음, 1.0보다 작거나 같음
  reasoning : str = Field(description = '분류 이유')

### 실습 (DetailedQueryAnalysis)

In [17]:
def analyze_query(query: str) -> QueryAnalysis:
  """LLM을 사용하여 쿼리를 분석하고 라우팅 경로를 결정"""
  system_prompt ="""당신은 쿼리를 잘 분석하고 라우팅 경로를 분류하는 전문가입니다. 주어진 쿼리를 분석해서 최적의 처리 경로를 분석하세요

  라우팅 경로:
  - direct : LLM이 자체 지식으로 답변 가능한 일반 질문(상식, 개념 설명)
  - rag : 회사 내부 문서나 특정 도메인 지식이 필요한 질문(정책, 매뉴얼, 사내 데이터)
  - web_search : 최신 정보나 실시간 데이터가 필요한 질문 (뉴스, 시세, 날씨)

  예시:
  - 파이썬의 데코레이터란? -> direct (일반 프로그래밍 지식)
  - 우리 회사 연차 규정은? -> rag (내부 문서 필요)
  - 오늘 날씨는? -> web_search(실시간 정보 필요)
  """

  # response = llm.invoke([SystemMessage(content = system_prompt), HumanMessage(content=query)])
  # 위와 같이 하면 "direct, direct 쿼리입니다..." 이런식으로 형식이 없으면 파싱하기 힘듦
    # 따라서 structured output으로 출력 형식을 강제하면 됨

  response = llm.with_structured_output(QueryAnalysis).invoke([
      SystemMessage(content = system_prompt),
      HumanMessage(content=query)
  ])

  return response





## Fallback & 라우팅 엔진

- 신뢰도(confidence)가 임계치 미만이면 fallback 경로(rag)로 전환
- `@dataclass`로 설정(RoutingConfig)과 로그(RoutingLog) 관리

In [19]:
result

QueryAnalysis(query='1+1은?', route='direct', confidence=1.0, reasoning='1+1은 기본적인 산수 문제로, LLM이 자체 지식으로 쉽게 답변할 수 있는 일반 질문입니다.')

In [21]:
# 실습

# pydantic 모델 -> 데이터 스키마를 받아 만드는 모델

# DetailedQueryAnalysis
# route, confidence, sub_topic (쿼리의 세부 주제를 나타내는 str 타입), language (쿼리 언어, ko, en, other)
# analyze_query_detailed 를 만들기


class DetailedQueryAnalysis(BaseModel):
  query : str = Field(description = '원본 쿼리')
  route : Literal["direct", "rag", "web_search"] = Field(
            description = "라우팅 경로: direct(직접 답변), rag(문서 검색), web_search(웹 검색)"
  )
  language :  Literal["ko", "en", "other"] = Field(
            description = "쿼리가 작성된 언어. ko(한국어), en(영어), other(기타 언어)"
  )
  sub_topic : str = Field(description = '쿼리의 세부 주제를 나타냄')
  confidence : float = Field(description = '분류 신뢰도(0.0 ~ 1.0)', ge=0.0, le=1.0)
  reasoning : str = Field(description = '분류 이유')

def analyze_query_detailed(query: str) -> DetailedQueryAnalysis:
    system_prompt = """당신은 쿼리를 상세히 분석하는 전문가입니다.
    주어진 쿼리에 대해 라우팅 경로, 신뢰도, 세부 주제, 그리고 언어를 판별하세요."""

    response = llm.with_structured_output(DetailedQueryAnalysis).invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ])
    return response

result = analyze_query_detailed("1+1은?")

In [22]:
result

DetailedQueryAnalysis(query='1+1은?', route='direct', language='ko', sub_topic='수학 문제', confidence=1.0, reasoning='주어진 쿼리는 기본적인 수학 문제로, 1+1의 결과를 묻고 있다. 단순하고 직관적인 질문이므로 직접적인 응답이 적합하다.')

## fallback

In [23]:
from dataclasses import dataclass, field
from datetime import datetime

In [24]:
@dataclass
class RoutingConfig:
  """라우팅 전략을 설정"""
  confidence_threshold : float = 0.7
  fallback_route : str = 'rag'
  enable_logging : bool = True
  max_retries : int = 2

@dataclass
class RoutingLog:
  """라우팅 로그"""
  timestamp : str
  query : str
  predicted_route : str
  actual_route : str
  confidence : float
  fallback_applied : bool

In [ ]:
# 만약 dataclass를 안쓴다면 추가 함수 항상 설정해줘야함
# class RoutingConfig:
  # def __init__(self, confidentce_threshold ....)
  # def ...

In [38]:
class RoutingEngine:
  def __init__(self, config: RoutingConfig = None):
    self.config = config or RoutingConfig()
    self.logs = [] # 로그 리스트 초기화

  def route(self, analysis: QueryAnalysis) -> str:
    """분석 결과를 바탕으로 최종 라우팅 경로를 결정"""
    predicted = analysis.route
    fallback_applied = False

    if analysis.confidence < self.config.confidence_threshold:
      actual = self.config.fallback_route
      fallback_applied = True
    else:
      actual = predicted

    if self.config.enable_logging:
      log = RoutingLog(timestamp = datetime.now().isoformat(), query = analysis.query,
                       predicted_route = predicted,
                       actual_route = actual,
                       confidence = analysis.confidence,
                       fallback_applied = fallback_applied)
      self.logs.append(log)

    return actual

  def get_stats(self) -> dict:
    if not self.logs:
      return {'total' : 0}
    total = len(self.logs)
    fallbacks = sum(1 for log in self.logs if log.fallback_applied)
    route_dist = {}

    for log in self.logs:
      route_dist[log.actual_route] = route_dist.get(log.actual_route, 0) + 1

    avg_conf = sum(log.confidence for log in self.logs) / total

    return {
        'total' : total,
        'fallback_rate' : fallbacks / total,
        'avg_confidence' : avg_conf,
        'route_distribution' : route_dist
    }

### RouteHandler (ABC 핸들러 패턴)

- 추상 클래스(`ABC`)로 공통 인터페이스 정의 → Direct / RAG / WebSearch 핸들러 구현
- `handlers` 딕셔너리로 라우팅 결과에 따라 핸들러 자동 선택

In [39]:
engine = RoutingEngine(RoutingConfig(confidence_threshold = 0.7))

In [40]:
analysis = analyze_query('파이썬 데코레이터가 뭔가요?')
final_route = engine.route(analysis)

In [41]:
final_route

'direct'

In [42]:
engine.logs

[RoutingLog(timestamp='2026-05-12T12:06:02.400951', query='파이썬 데코레이터가 뭔가요?', predicted_route='direct', actual_route='direct', confidence=0.9, fallback_applied=False)]

### 가중치 기반 라우팅 (AdvancedRoutingConfig)

In [43]:
from abc import ABC, abstractmethod # Abstract Base Class 추상 클래스

## LangGraph 기반 Adaptive RAG

- `AdaptiveRAGState`: query, route, confidence, response, documents, search_results 채널
- `query_analyzer_node` → `route_query` 조건부 분기 → direct / rag / web_search 노드

In [50]:
class RouteHandler(ABC):
  @abstractmethod # 해당 데코레이터로 선언된 함수들을 구현해야 한다는 의미?
  def handle(self, query:str) -> str:
    pass

In [44]:
# DirectHandler
# RAGHandler

In [51]:
class DirectHandler(RouteHandler):
  """LLM이 직접 답변하는 경우"""
  def handle(self, query:str) -> str:
    response = llm.invoke(
        [HumanMessage(content=query)]
    )
    return response.content


class RAGHandler():
  """RAG 검색해서 답변하는 경우"""
  def handle(self, query:str) -> str:
    return f"RAG '{query}'에 대한 문서 검색 결과 입니다."

class WebSearchHandler(RouteHandler):
  """웹 검색 후 답변하는 경우"""

  def handle(self, query:str) -> str:
    return f"WebSearch '{query}'에 대한 문서 검색 결과입니다."

In [52]:
handlers = {
    'direct' : DirectHandler(),
    'rag' : RAGHandler(),
    'web_search' : WebSearchHandler()

}

test_q = '파이썬에서 리스트 컴프리헨션이란?'
analysis = analyze_query(test_q)
route = engine.route(analysis)
result = handlers[route].handle(test_q)
print(result)

리스트 컴프리헨션(List Comprehension)은 파이썬에서 리스트를 간결하고 효율적으로 생성하는 방법입니다. 기존의 리스트를 기반으로 새로운 리스트를 만들거나, 특정 조건에 맞는 요소들을 필터링하는 데 사용됩니다. 리스트 컴프리헨션은 일반적으로 for 루프와 if 문을 결합하여 표현합니다.

기본적인 구조는 다음과 같습니다:

```python
[표현식 for 항목 in iterable if 조건]
```

여기서:
- `표현식`은 각 항목에 대해 수행할 작업을 정의합니다.
- `항목`은 iterable(반복 가능한 객체)의 개별 요소입니다.
- `iterable`은 리스트, 튜플, 문자열 등 반복 가능한 객체를 의미합니다.
- `조건`은 (선택적) 각 항목이 포함될지 여부를 결정하는 필터 기능입니다.

### 예시

1. 기본적인 리스트 컴프리헨션:

```python
squares = [x**2 for x in range(10)]
print(squares)  # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
```

2. 조건을 포함한 리스트 컴프리헨션:

```python
even_squares = [x**2 for x in range(10) if x % 2 == 0]
print(even_squares)  # [0, 4, 16, 36, 64]
```

3. 중첩된 리스트 컴프리헨션을 사용한 예:

```python
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flattened = [num for row in matrix for num in row]
print(flattened)  # [1, 2, 3, 4, 5, 6, 7, 8, 9]
```

리스트 컴프리헨션은 코드의 가독성을 높이고, 보다 간결하게 리스트를 생성할 수 있도록 도와줍니다. 그러나 너무 복잡한 표현을 사용할 경우 오히려 가독성이 떨어질 수 있으므로 적절히 사용하는 것이 중요합니다.


In [57]:
# 실습, 라우팅 경로 가중치
# AdvancedRoutingConfig 추가 -> route_weights (direct : 1.0, rag : 0.8, web_search: 0.6)
# weighted_route() 0.8
# confidence_threshold : 0.5 이하 -> fallback(rag)

@dataclass
class AdvancedRoutingConfig:
  """가중치 기반 라우팅 전략을 설정"""
  confidence_threshold : float = 0.5
  route_weights : dict= field(default_factory = lambda : {'direct' : 1.0, 'rag' : 0.8, 'web_search': 0.6})
  fallback_route : str = 'rag'
  enable_logging : bool = True
  max_retries : int = 2


class WeightedRoutingEngine:
    def __init__(self, config: AdvancedRoutingConfig = None): # init을 __init__으로 수정
        self.config = config or AdvancedRoutingConfig()

    def weighted_route(self, analysis: QueryAnalysis) -> dict:
        weight = self.config.route_weights.get(analysis.route, 1.0)
        weighted_score = analysis.confidence * weight
        if weighted_score < self.config.confidence_threshold:
            final_route = self.config.fallback_route
            fallback = True
        else:
            final_route = analysis.route
            fallback = False
        return {
            "query": analysis.query[:25],
            "predicted": analysis.route,
            "confidence": analysis.confidence,
            "weight": weight,
            "weighted_score": round(weighted_score, 3),
            "final_route": final_route,
            "fallback": fallback
        }

### 실습 (fallback_node 추가)

- confidence < 0.5 → "질문을 더 구체적으로 해주세요" 반환
- `route_query_with_fallback`으로 fallback 경로 추가

In [58]:
weighted_engine = WeightedRoutingEngine()
test_q = '사내 보안 교육이 언제인가요?'

analysis = analyze_query(test_q)
result = weighted_engine.weighted_route(analysis)
print(result)

{'query': '사내 보안 교육이 언제인가요?', 'predicted': 'rag', 'confidence': 0.9, 'weight': 0.8, 'weighted_score': 0.72, 'final_route': 'rag', 'fallback': False}


In [59]:
# 랭그래프 -> Adaptive RAG 구현하기 좋음
  # 1. 상태관리 용이: 노드 함수간 상태값, 타입 관리
  # 2. 그래프 시각화, 연결상태 등

class AdaptiveRAGState(TypedDict):
  query: int
  route : str
  confidence : float
  reasoning : str
  response : str
  documents : list
  search_results : list


In [61]:
# 쿼리가 들어오면 쿼리를 분석해서 라우팅 경로를 결정
def query_analyzer_node(state: AdaptiveRAGState) -> dict:
  """쿼리 분석 노드"""
  query = state['query']
  analysis = analyze_query(query)
  return {'route': analysis.route, 'confidence': analysis.confidence, 'reasoning': analysis.reasoning}

def direct_answer_node(state : AdaptiveRAGState) -> dict:
  """LLM 직접 답변 노드"""
  query = state['query']
  response = llm.invoke([HumanMessage(content=query)])
  return {'response': f"[Direct] {response.content}"}

def rag_search_node(state : AdaptiveRAGState) -> dict:
  """RAG 검색 노드"""
  query = state['query']
  return {'documents': [f"[Direct] {query}에 대한 검색 문서"], 'respose': f"[RAG] '{query}'에 대한 문서 기반 답변"}

def web_search_node(state : AdaptiveRAGState) -> dict:
  """웹 검색 노드"""
  query = state['query']
  return {'search_results': [f"[Web] {query} 검색 결과"],
          'response': f"[WebSearch] '{query}'에 대한 웹 검색 답변"}

In [63]:
def route_query(state: AdaptiveRAGState) -> dict:
  route = state['route']
  confidence = state['confidence']

  if confidence < 0.7:
    return 'rag_search'

  if route == 'direct':
    return 'direct_answer'

  elif route == 'rag':
    return 'rag_search'

  elif route == 'web_search'
    return 'web_search'
  else:
    return 'rag_search'

In [68]:
workflow = StateGraph(AdaptiveRAGState)
workflow.add_node('query_analyzer', query_analyzer_node)
workflow.add_node('direct_answer', direct_answer_node)
workflow.add_node('rag_search', rag_search_node)
workflow.add_node('web_search', web_search_node)

workflow.add_edge(START, 'query_analyzer')
workflow.add_conditional_edges(
    'query_analyzer',
    route_query,
    {
      'direct_answer' : 'direct_answer',
      'rag_search' : 'rag_search',
      'web_search' : 'web_search',

    }

)
workflow.add_edge('direct_answer', END)
workflow.add_edge('rag_search', END)
workflow.add_edge('web_search', END)

app = workflow.compile()

In [69]:
test_q = '우리 회사 재택근무 정책은'

result = app.invoke({
  'query': test_q,
  'route' : "",
  'confidence' : 0.0,
  'reasoning' : "",
  'response' : "",
  'documents' : [],
  'search_results' : []

})

In [70]:
result

{'query': '우리 회사 재택근무 정책은',
 'route': 'rag',
 'confidence': 0.95,
 'reasoning': '회사의 특정 내부 문서나 정책을 필요로 하는 질문이므로, rag 경로로 분류되었다.',
 'response': '',
 'documents': ['[Direct] 우리 회사 재택근무 정책은에 대한 검색 문서'],
 'search_results': []}

In [ ]:
# 실습
# route query 함수 사용
# 1. fallback_node 추가 : confidence < 0.5면 사용자에게 재질문 요청, return 질문을 더 구체적으로 해주세요
# 2. route_query : 0.5보다 작으면 fallback
# workflow 재구성

In [78]:
def fallback_node(state: AdaptiveRAGState) -> dict:
    """confidence가 낮을 때 다시 질문"""
    return "질문을 더 구체적으로 해주세요"

def route_query_with_fallback(state: AdaptiveRAGState) -> str:
    route = state['route']
    confidence = state['confidence']

    if confidence < 0.5:
        return "fallback"

    if route == 'direct':
        return 'direct_answer'
    elif route == 'rag':
        return 'rag_search'
    elif route == 'web_search':
        return 'web_search'
    else:
        return 'rag_search'

In [79]:
workflow = StateGraph(AdaptiveRAGState)
workflow.add_node('query_analyzer', query_analyzer_node)
workflow.add_node('direct_answer', direct_answer_node)
workflow.add_node('rag_search', rag_search_node)
workflow.add_node('web_search', web_search_node)
workflow.add_node('fallback', fallback_node)

workflow.add_edge(START, 'query_analyzer')
workflow.add_conditional_edges(
    'query_analyzer',
    route_query_with_fallback,
    {
      'direct_answer' : 'direct_answer',
      'rag_search' : 'rag_search',
      'web_search' : 'web_search',
      'fallback' : 'fallback'

    }

)
workflow.add_edge('direct_answer', END)
workflow.add_edge('rag_search', END)
workflow.add_edge('web_search', END)
workflow.add_edge('fallback', END)

app = workflow.compile()

In [84]:
test_q = 'ㅁㄴㄻ 3 '

result = app.invoke({
  'query': test_q,
  'route' : "",
  'confidence' : 0.0,
  'reasoning' : "",
  'response' : "",
  'documents' : [],
  'search_results' : []

})

InvalidUpdateError: Expected dict, got 질문을 더 구체적으로 해주세요
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [83]:
result

{'query': '우리',
 'route': 'rag',
 'confidence': 0.9,
 'reasoning': '이는 특정 회사 또는 조직의 활동이나 정책에 관한 질문일 가능성이 높으며, 내부 문서나 특정 지식이 필요할 것으로 보입니다.',
 'response': '',
 'documents': ['[Direct] 우리에 대한 검색 문서'],
 'search_results': []}